In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import bitsandbytes
from PIL import Image
from IPython.display import display
from pathlib import Path
import unicodedata
import sys
import torch
import open_clip
import chromadb
import re
import os
import numpy as np
import subprocess
from PIL import Image
from pocket_tts import TTSModel
import scipy.io.wavfile
from scipy.io import wavfile
import soundfile as sf
from datetime import timedelta

from sqlalchemy.orm import Session
from sqlalchemy import text
import json
import boto3
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor
import pillow_avif
import shutil
import hashlib

c:\Users\georg\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
root = Path.cwd()
while not (root / "src").exists():
    root = root.parent

sys.path.append(str(root))
from src.db.session import engine
from src.models import Pharaoh

In [4]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(torch.version.cuda)
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.5.1+cu121
12.1
CUDA available: True


In [5]:
!nvidia-smi

Wed Mar 25 20:29:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   52C    P8             14W /   85W |    4178MiB /   6144MiB |     35%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)
model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [7]:
NAME = "Ramesses II"
is_landmark = False
if is_landmark:
    with Session(engine) as session:
        result = session.execute(text("SELECT landmark_script FROM landmarks_scripts as ls, landmarks as l WHERE ls.landmark_id = l.id AND l.name = :name"), {"name": NAME})
        script = result.fetchall()[0][0]

else:
    with Session(engine) as session:
        result = session.execute(text("SELECT pharaoh_script FROM pharaohs_scripts as ps, pharaohs as p WHERE ps.pharaoh_id = p.id AND p.name = :name"), {"name": NAME})
        script = result.    fetchall()[0][0]

print(script)

Rameses II was one of ancient Egypt’s greatest pharaohs. Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war. At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side. He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.

Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region. This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns. Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.

He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength. Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first known peace treaty between two great powers.

Ruling 

In [8]:
paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
paragraphs

['Rameses II was one of ancient Egypt’s greatest pharaohs. Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war. At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side. He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.',
 'Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region. This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns. Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.',
 "He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength. Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first known peace treaty between two great powers."

In [9]:
sentences = []
sentences_per_paragraph = []
for p in paragraphs:
    sentences.append([s.strip() for s in re.split(r'(?<=[.!?]) +', p) if s.strip()])
    sentences_per_paragraph.append(len(sentences[-1]))
sentences

[['Rameses II was one of ancient Egypt’s greatest pharaohs.',
  'Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war.',
  'At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.',
  'He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.'],
 ['Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region.',
  'This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns.',
  'Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.'],
 ["He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength.",
  'Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first known peac

### Audio

In [10]:
#Text-to-Speech
tts_model = TTSModel.load_model()
voice_state = tts_model.get_state_for_audio_prompt("alba")

i = 0
for p in sentences:
    for s in p:
        audio = tts_model.generate_audio(voice_state, s)
        scipy.io.wavfile.write(f"Outputs\\output_{i}.wav", tts_model.sample_rate, audio.numpy())
        i += 1

In [11]:
wav_files = sorted([f for f in os.listdir("Outputs") if f.endswith('.wav')])
print(wav_files)

audio_data = []
seconds_per_paragraph = []
samplerate = None

for i in range(len(wav_files)):
    file_name = f"Outputs\\output_{i}.wav"
    
    data, sr = sf.read(file_name)
    
    if samplerate is None:
        samplerate = sr
    elif sr != samplerate:
        raise ValueError("Sample rates do not match!")

    audio_data.append(data)

# Concatenate along time axis
combined = np.concatenate(audio_data, axis=0)

# Write output (keeps float format safely)
sf.write("Outputs\\Final audio.wav", combined, samplerate)

['output_0.wav', 'output_1.wav', 'output_10.wav', 'output_11.wav', 'output_2.wav', 'output_3.wav', 'output_4.wav', 'output_5.wav', 'output_6.wav', 'output_7.wav', 'output_8.wav', 'output_9.wav']


In [12]:
durations = []
sentences_durations = []
images_needed = []
#seconds = []
i = 0
for p in sentences:
    duration_seconds = 0
    for s in p:
        file_path = f"Outputs\\output_{i}.wav"
        # Returns the sample rate (Fs) and the data as a NumPy array
        Fs, data = wavfile.read(file_path) 
        # The length of the data array divided by the sample rate gives the duration
        sentence_duration = len(data) / float(Fs)
        duration_seconds += sentence_duration
        sentences_durations.append(sentence_duration)
        i += 1
        os.remove(file_path)
    durations.append(duration_seconds)
    images_needed.append(max(1, int(duration_seconds / 6)))
    '''    section_seconds = duration_seconds / images_needed[-1]
    for img in range(images_needed[-1]):
        seconds.append(section_seconds)'''


print(f"Durations of audio files: {durations}")
print(f"Images needed for each paragraph: {images_needed}")
print(f"Durations of sentences: {sentences_durations}")
#print(f"Seconds for each image: {seconds}")

Durations of audio files: [25.92, 20.48, 15.2, 21.28]
Images needed for each paragraph: [4, 3, 2, 3]
Durations of sentences: [4.32, 8.56, 8.4, 4.64, 5.84, 7.76, 6.88, 7.44, 7.76, 10.8, 5.44, 5.04]


In [13]:
i = 0
sentence_start = 0
sentence_end = 0
image_text_chunks = []
seconds_for_chunk = []

# Use the already-split sentences from the first pass (list of lists)
for para_idx, para_sentences in enumerate(sentences_per_paragraph):
    # sentences_split is your original `sentences` list of lists
    para_sentence_list = sentences[para_idx]
    images_for_paragraph = images_needed[para_idx]
    images_for_paragraph = min(images_for_paragraph, len(para_sentence_list))

    total_sentences = len(para_sentence_list)
    base = total_sentences // images_for_paragraph
    remainder = total_sentences % images_for_paragraph

    groups = []
    start = 0
    for img_idx in range(images_for_paragraph):
        extra = 1 if img_idx < remainder else 0
        end = start + base + extra

        chunk_sentence_end = sentence_start + end  # absolute index into sentences_durations
        chunk_duration = sum(sentences_durations[sentence_start + start : sentence_start + end])
        seconds_for_chunk.append(chunk_duration)

        chunk = " ".join(para_sentence_list[start:end])
        groups.append(chunk)

        start = end

    sentence_start += total_sentences  # advance global pointer by paragraph's sentence count
    image_text_chunks.append(groups)

print(f"Text chunks for images: {image_text_chunks}")
print(f"Seconds for each chunk: {seconds_for_chunk}")

Text chunks for images: [['Rameses II was one of ancient Egypt’s greatest pharaohs.', 'Born to Seti I and Queen Tuya, he accompanied his father on military campaigns in Libya and Palestine during his youth, learning leadership through war.', 'At twenty-two, Rameses launched a Nubian campaign with his sons Khaemweset and Amunhirwenemef by his side.', 'He also fought wars in Canaan, bringing back treasures and prisoners to enrich Egypt.'], ['Rameses built the grand capital of Per-Ramesses near Avaris in the eastern Delta region.', 'This city served as a palace, administrative center, armory, military stable, and base for foreign campaigns.', 'Its walls were adorned with faience tiles, statues, balconies, doorways, and elaborate throne rooms.'], ["He continued his father’s restoration projects throughout Egypt and Nubia, reopening mines and quarries to ensure the empire's strength.", 'Rameses faced a legendary battle against the Hittites that ended in stalemate but led to history’s first 

### Retrieval Approaches

In [14]:
def cosine(a,b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

In [47]:
import ast
def retrieve_images_for_chunks(NAME):
    fetched_images_ids = []
    fetched_images_paths = []
    repeated_images_ids = []
    scores = []
    trials = []
    for paragraph_chunks in image_text_chunks:
        for chunk in paragraph_chunks:
            text_tokens = open_clip.get_tokenizer()([chunk]).to(device)
        
            with torch.no_grad():
                text_embedding = model.encode_text(text_tokens)
                text_embedding /= text_embedding.norm(dim=-1, keepdim=True)

            embedding_list = text_embedding.cpu().numpy().tolist()[0]
            if is_landmark:
                query = "SELECT li.id, li.image_path, li.image_embedding::text FROM landmark_images as li JOIN landmarks as l ON li.landmark_id = l.id WHERE l.name = :name ORDER BY image_embedding <=> CAST(:embedding AS vector)"
            else:
                query = "SELECT pi.id, pi.image_path, pi.image_embedding::text FROM pharaohs_images as pi JOIN pharaohs as p ON pi.pharaoh_id = p.id where p.name = :name ORDER BY image_embedding <=> CAST(:embedding AS vector)"


            with Session(engine) as session:
                result = session.execute(text(query), {"name": NAME, "embedding": embedding_list})
                images = result.fetchall()
                image_id = images[0][0]
                image_path = images[0][1]
                image_embedding = images[0][2]
                image_embedding = np.fromstring(image_embedding.strip("[]"), sep=",", dtype=np.float32)
                j = 0
                while image_id in fetched_images_ids:
                    images.pop(0)
                    if not images:
                        #print("No more unique images available for this chunk. Duplicate images will be used.")
                        result = session.execute(text(query), {"name": NAME, "embedding": embedding_list})
                        images = result.fetchall()
                        image_id = images[0][0]
                        while image_id in repeated_images_ids:
                            images.pop(0)
                            image_id = images[0][0]
                            image_path = images[0][1]
                            image_embedding = images[0][2]
                            image_embedding = np.fromstring(image_embedding.strip("[]"), sep=",", dtype=np.float32)
                            j += 1
                        repeated_images_ids.append(image_id)
                        break
                    image_id = images[0][0]
                    image_path = images[0][1]
                    image_embedding = images[0][2]
                    image_embedding = np.fromstring(image_embedding.strip("[]"), sep=",", dtype=np.float32)
                    j += 1
                score = cosine(text_embedding.cpu().numpy()[0], image_embedding)
                scores.append(score)
                fetched_images_ids.append(image_id)
                fetched_images_paths.append(image_path)
                trials.append(j)
            
        
            #print(f"Text chunk: {chunk}")
            #print(f"Image ID: {image_id} on trial {j}")
            #print(f"image path: {image_path}")
    average_score = sum(scores) / len(scores) if scores else 0
    average_trials = sum(trials) / len(trials) if trials else 0
    return fetched_images_ids, fetched_images_paths, average_score, average_trials

In [15]:
fetched_images_ids, fetched_images_paths, score, trial = retrieve_images_for_chunks()

In [15]:
IMAGE_SIM_WEIGHT = 0.3
DESC_SIM_WEIGHT = 0.7

def cosine(a,b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

In [49]:
def retrieve_images_semantic(NAME):
    fetched_images_ids = []
    fetched_images_paths = []
    repeated_images_ids = []
    best_scores = []
    trials = []
    with Session(engine) as session:
        result = session.execute(
            text("""
                SELECT 
                    pi.id,
                    pi.image_path,
                    pi.image_embedding,
                    pi.image_description
                FROM pharaohs_images pi
                JOIN pharaohs p ON pi.pharaoh_id = p.id
                WHERE p.name = :name
            """),
            {"name": NAME}
        )

        images_data=result.fetchall()
        
    clip_tokenizer = open_clip.get_tokenizer("ViT-H-14")
    processed_images=[]
    for row in images_data:
        image_id,image_path,image_embedding,image_description=row
        if isinstance(image_embedding,str):
            image_embedding=json.loads(image_embedding)

        image_embedding=np.array(image_embedding)
        image_embedding=image_embedding/np.linalg.norm(image_embedding)

        tokens=clip_tokenizer([image_description]).to(device)

        with torch.no_grad():
            desc_emb=model.encode_text(tokens)
            desc_emb/=desc_emb.norm(dim=-1,keepdim=True)

        desc_emb=desc_emb.cpu().numpy()[0]

        processed_images.append({
            "id": image_id,
            "path": image_path,
            "img_emb": image_embedding,
            "desc_emb": desc_emb
        })
    
    for paragraph_chunks in image_text_chunks:
        for chunk in paragraph_chunks:
            text_tokens = open_clip.get_tokenizer()([chunk]).to(device)
            with torch.no_grad():
                emb = model.encode_text(text_tokens)
                emb /= emb.norm(dim=-1, keepdim=True)
            scene_emb = emb.cpu().numpy()[0]

            ranked = []
            for img in processed_images:
                image_sim = cosine(scene_emb, img["img_emb"])
                desc_sim = cosine(scene_emb, img["desc_emb"])
                score = IMAGE_SIM_WEIGHT * image_sim + DESC_SIM_WEIGHT * desc_sim
                ranked.append((score, img["id"], img["path"]))
                
            ranked.sort(reverse=True, key=lambda x: x[0])
            ranked2= ranked.copy()
            best_score, best_id, best_path = ranked[0]
            j = 0
            while best_id in fetched_images_ids:
                ranked.pop(0)
                j += 1
                if not ranked:
                    #print("No more unique images available for this chunk. Duplicate images will be used.")
                    best_score, best_id, best_path = ranked2[0]
                    while best_id in repeated_images_ids:
                        ranked2.pop(0)
                        j += 1
                        if not ranked2:
                            print("No more unique images available at all.")
                            break
                        best_score, best_id, best_path = ranked2[0]
                    break
                best_score, best_id, best_path = ranked[0]
            fetched_images_ids.append(best_id)
            fetched_images_paths.append(best_path)
            best_scores.append(best_score)
            trials.append(j)
            #print(f"Text chunk: {chunk}")
            #print(f"Best image ID: {best_id} with score {best_score}")
            #print(f"Best image path: {best_path}")
    average_score = sum(best_scores) / len(best_scores) if best_scores else 0
    average_trials = sum(trials) / len(trials) if trials else 0
    return fetched_images_ids, fetched_images_paths, average_score, average_trials

In [19]:
fetched_images_ids, fetched_images_paths, score, trials = retrieve_images_semantic()

In [55]:
import time
from openai import OpenAI

client = OpenAI(
    api_key="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",
    base_url="https://api.groq.com/openai/v1"
)

def generate_visual_prompt(scene_text):

    prompt = f"""
You are an expert visual prompt generator for an Ancient Egypt image retrieval system.

Convert the narration into a SHORT visual description of what an image should show, that will help retrieve the best matching historical image from an Ancient Egypt dataset.


The image database contains:
- Pharaoh statues
- Egyptian gods and goddesses
- Royal family members (father, son, wife, daughter)
- Temple relief carvings
- Archaeological artifacts
- Monuments and temples

Guidelines:
1. Identify the MAIN ENTITY (pharaoh, god, goddess, royal family).
2. Include the NAME if present.
3. Include VISUAL ELEMENTS such as: statue, temple relief, wall carving, hieroglyphs, monument, artifact.
4. Include LOCATION if mentioned (temple name, city, monument).
5. Prefer descriptions that look like museum or archaeological images.
6. Use concise keyword-style phrases, not storytelling.

Output a single visual description.

Narration:
{scene_text}

Visual prompt:
"""

    time.sleep(0.5)
    response = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[
            {"role": "system", "content": "You create visual prompts for documentary scenes."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content.strip()

def retrieve_images_image_oriental(NAME):
    fetched_images_ids = []
    fetched_images_paths = []
    repeated_images_ids = []
    best_scores = []
    trials = []
    with Session(engine) as session:
        result = session.execute(
            text("""
                SELECT 
                    pi.id,
                    pi.image_path,
                    pi.image_embedding,
                    pi.image_description
                FROM pharaohs_images pi
                JOIN pharaohs p ON pi.pharaoh_id = p.id
                WHERE p.name = :name
            """),
            {"name": NAME}
        )

        images_data=result.fetchall()

    clip_tokenizer = open_clip.get_tokenizer("ViT-H-14")
    processed_images=[]
    for row in images_data:
        image_id,image_path,image_embedding,image_description=row
        if isinstance(image_embedding,str):
            image_embedding=json.loads(image_embedding)

        image_embedding=np.array(image_embedding)
        image_embedding=image_embedding/np.linalg.norm(image_embedding)

        tokens=clip_tokenizer([image_description]).to(device)

        with torch.no_grad():
            desc_emb=model.encode_text(tokens)
            desc_emb/=desc_emb.norm(dim=-1,keepdim=True)

        desc_emb=desc_emb.cpu().numpy()[0]

        processed_images.append({
            "id": image_id,
            "path": image_path,
            "img_emb": image_embedding,
            "desc_emb": desc_emb
        })
    for paragraph_chunks in image_text_chunks:
        for chunk in paragraph_chunks:
            visual_prompt = generate_visual_prompt(chunk)
            prompt_tokens = clip_tokenizer([visual_prompt]).to(device)
            with torch.no_grad():
                prompt_emb = model.encode_text(prompt_tokens)
                prompt_emb /= prompt_emb.norm(dim=-1, keepdim=True)
            prompt_emb = prompt_emb.cpu().numpy()[0]

            text_tokens = clip_tokenizer([chunk]).to(device)
            with torch.no_grad():
                text_emb = model.encode_text(text_tokens)
                text_emb /= text_emb.norm(dim=-1, keepdim=True)
            text_emb = text_emb.cpu().numpy()[0]

            ranked = []
            for img in processed_images:
                image_sim = cosine(prompt_emb, img["img_emb"])
                desc_sim = cosine(text_emb, img["desc_emb"])
                score = IMAGE_SIM_WEIGHT * image_sim + DESC_SIM_WEIGHT * desc_sim
                ranked.append((score, img["id"], img["path"]))
            ranked.sort(reverse=True, key=lambda x: x[0])

            ranked2= ranked.copy()
            best_score, best_id, best_path = ranked[0]
            j = 0
            while best_id in fetched_images_ids:
                ranked.pop(0)
                j += 1
                if not ranked:
                    #print("No more unique images available for this chunk. Duplicate images will be used.")
                    best_score, best_id, best_path = ranked2[0]
                    while best_id in repeated_images_ids:
                        ranked2.pop(0)
                        j += 1
                        if not ranked2:
                            print("No more unique images available at all.")
                            break
                        best_score, best_id, best_path = ranked2[0]
                    break
                best_score, best_id, best_path = ranked[0]
            fetched_images_ids.append(best_id)
            fetched_images_paths.append(best_path)
            best_scores.append(best_score)
            trials.append(j)
            #print(f"Text chunk: {chunk}")
            #print(f"Best image ID: {best_id} with score {best_score}")
            #print(f"Best image path: {best_path}")
    average_score = sum(best_scores) / len(best_scores) if best_scores else 0
    average_trials = sum(trials) / len(trials) if trials else 0
    return fetched_images_ids, fetched_images_paths, average_score, average_trials

In [ ]:
fetched_images_ids, fetched_images_paths, best_scores = retrieve_images_image_oriental()

### Retrieval Evaluation

In [43]:
with Session(engine) as session:
        result = session.execute(text("select p.name from pharaohs_scripts as ps, pharaohs as p where p.id = ps.id"))
        names = [row[0] for row in result.fetchall()]
names.remove('Amenhotep III and God Re-Horakhty')
names.remove('Amenhotep III and Wife Queen Tiye')
names.remove('Amun and Mut (God & Goddess)')
names.remove('Hor (son of Ankh-Khonsu)')
names.remove('Itysen (Prince, probably son of Djedefre)')
names.remove('Isis (Goddess) with her child')
names.remove('Menkaure and his wife (likely Queen Khamerernebty II)')
names.remove('Menkaure with Goddesses Hathor and Bat')
names.remove('Merenptah and Goddess Mut')
names.remove('Ramesses II and God Ptah and Goddess Sekhment')
names.remove('Nectanebo II and God Horus')
names.remove('Osirian Triad and a King (Osiris, Isis, Horus, and Hormheb)')
names.remove('Rameses II and Hauron')
names.remove('Ramesess II and God Amun & Goddess Mut')
names.remove('Ramesses II and God Ptah')
names.remove('Ramesses II and God Ptah and a Goddess')
names.remove('Ramesses II and Goddess')
names.remove('Ramesses II and Goddess Anath')
names.remove('Ramesses II and Goddesses Isis and Hathor')
names.remove('Ramesses III and Gods Horus and Seth')
names.remove('Hauron God')
names

['Achoris',
 'Ahmose I',
 'Akhenaton',
 'Alexander The Great',
 'Amasis II',
 'Amenemhat I',
 'Amenemhat III',
 'Amenhotep II',
 'Amenhotep III',
 'Amenirdis (Daughter of Kashta)',
 'Amun (God)',
 'Amun-Ra (God)',
 'Anath Goddess',
 'Ankhsenamun',
 'Anubis (God)',
 'Arsinoe II (Queen, sister and wife of Ptolemy IV)',
 'Arsinoe III (Queen, daughter of Ptolemy I and wife of Ptolemy II)',
 'Bat Goddess',
 'Cleopatra VII Philopator',
 'Djoser',
 'Hathor (Goddess)',
 'Hatshepsut',
 'Hor I',
 'Horemheb',
 'Horus (God)',
 'Isis (Goddess)',
 'Isis (mother of Thutmose III)',
 'Khamerernebty II',
 'Khafre',
 'Khasekhem',
 'Khonsu (God)',
 'Khufu',
 'Menkuare',
 'Mentuhotep II',
 'Merenptah',
 'Meresankh (Queen, wife of Khafre)',
 'Mut (Goddess)',
 'Mutnofret (Queen, Wife of Thutmose I and mother of Thutmose II)',
 'Nectanebo I',
 'Nectanebo II',
 'Nefertiti',
 'Nofret (Queen, possibly Nofret II, wife of Senusret II)',
 'Niuserre',
 'Osiris (God)',
 'Osorkon II',
 'Pepi I',
 'Psusennes I',
 'Ptah

In [48]:
scores = []
trials = []
for name in names:
    with Session(engine) as session:
        result = session.execute(text("SELECT pharaoh_script FROM pharaohs_scripts as ps, pharaohs as p WHERE ps.pharaoh_id = p.id AND p.name = :name"), {"name": name})
        script = result.fetchall()[0][0]

    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
    sentences = []
    sentences_per_paragraph = []
    for p in paragraphs:
        sentences.append([s.strip() for s in re.split(r'(?<=[.!?]) +', p) if s.strip()])
        sentences_per_paragraph.append(len(sentences[-1]))
        
    best_images_ids, best_images_paths, average_score, average_trials = retrieve_images_for_chunks(name)
    scores.append(average_score)
    trials.append(average_trials)
    print(f"Pharaoh: {name}, Average CLIP score: {scores[-1]}, Average retrieval trials: {trials[-1]}")
print(f"Scores for each pharaoh: {dict(zip(names, scores))}")
print(f"Overall average score for chunks approach: {sum(scores) / len(scores) if scores else 0}")
print(f"Overall average retrieval trials for chunks approach: {sum(trials) / len(trials) if trials else 0}")

Pharaoh: Achoris, Average CLIP score: 0.23382891714572906, Average retrieval trials: 3.4166666666666665
Pharaoh: Ahmose I, Average CLIP score: 0.25675880908966064, Average retrieval trials: 2.4166666666666665
Pharaoh: Akhenaton, Average CLIP score: 0.2742162048816681, Average retrieval trials: 1.4166666666666667
Pharaoh: Alexander The Great, Average CLIP score: 0.22664785385131836, Average retrieval trials: 3.75
Pharaoh: Amasis II, Average CLIP score: 0.2341011017560959, Average retrieval trials: 3.25
Pharaoh: Amenemhat I, Average CLIP score: 0.26321345567703247, Average retrieval trials: 2.25
Pharaoh: Amenemhat III, Average CLIP score: 0.26488155126571655, Average retrieval trials: 2.3333333333333335
Pharaoh: Amenhotep II, Average CLIP score: 0.251228004693985, Average retrieval trials: 2.9166666666666665
Pharaoh: Amenhotep III, Average CLIP score: 0.2769683003425598, Average retrieval trials: 1.5
Pharaoh: Amenirdis (Daughter of Kashta), Average CLIP score: 0.2349942922592163, Average

In [50]:
scores = []
trials = []
for name in names:
    with Session(engine) as session:
        result = session.execute(text("SELECT pharaoh_script FROM pharaohs_scripts as ps, pharaohs as p WHERE ps.pharaoh_id = p.id AND p.name = :name"), {"name": name})
        script = result.fetchall()[0][0]

    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
    sentences = []
    sentences_per_paragraph = []
    for p in paragraphs:
        sentences.append([s.strip() for s in re.split(r'(?<=[.!?]) +', p) if s.strip()])
        sentences_per_paragraph.append(len(sentences[-1]))
        
    best_images_ids, best_images_paths, average_score, average_trials = retrieve_images_semantic(name)
    scores.append(average_score)
    trials.append(average_trials)
    print(f"Pharaoh: {name}, Average CLIP score: {scores[-1]}, Average retrieval trials: {trials[-1]}")
print(f"Averages for each pharaoh: {dict(zip(names, scores))}")
print(f"Overall average score for semantic approach: {sum(scores) / len(scores) if scores else 0}")
print(f"Overall average retrieval trials for semantic approach: {sum(trials) / len(trials) if trials else 0}")

Pharaoh: Achoris, Average CLIP score: 0.3322843442467156, Average retrieval trials: 4.333333333333333
Pharaoh: Ahmose I, Average CLIP score: 0.40120287670620897, Average retrieval trials: 1.75
Pharaoh: Akhenaton, Average CLIP score: 0.3615270674593907, Average retrieval trials: 3.0833333333333335
Pharaoh: Alexander The Great, Average CLIP score: 0.3427146516550097, Average retrieval trials: 2.5833333333333335
Pharaoh: Amasis II, Average CLIP score: 0.3591540892910972, Average retrieval trials: 3.9166666666666665
Pharaoh: Amenemhat I, Average CLIP score: 0.38436145533855476, Average retrieval trials: 2.3333333333333335
Pharaoh: Amenemhat III, Average CLIP score: 0.3683940179388739, Average retrieval trials: 2.0833333333333335
Pharaoh: Amenhotep II, Average CLIP score: 0.38143150365721795, Average retrieval trials: 2.9166666666666665
Pharaoh: Amenhotep III, Average CLIP score: 0.4007703776625029, Average retrieval trials: 2.0833333333333335
Pharaoh: Amenirdis (Daughter of Kashta), Averag

In [56]:
scores = []
trials = []
for name in names:
    with Session(engine) as session:
        result = session.execute(text("SELECT pharaoh_script FROM pharaohs_scripts as ps, pharaohs as p WHERE ps.pharaoh_id = p.id AND p.name = :name"), {"name": name})
        script = result.fetchall()[0][0]

    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', script) if p.strip()]
    sentences = []
    sentences_per_paragraph = []
    for p in paragraphs:
        sentences.append([s.strip() for s in re.split(r'(?<=[.!?]) +', p) if s.strip()])
        sentences_per_paragraph.append(len(sentences[-1]))
        
    best_images_ids, best_images_paths, average_score, average_trials = retrieve_images_image_oriental(name)
    scores.append(average_score)
    trials.append(average_trials)
    print(f"Pharaoh: {name}, Average CLIP score: {scores[-1]}")
print(f"Averages for each pharaoh: {dict(zip(names, scores))}")
print(f"Overall average score for image-oriented approach: {sum(scores) / len(scores) if scores else 0}")
print(f"Overall average retrieval trials for image-oriented approach: {sum(trials) / len(trials) if trials else 0}")

Pharaoh: Achoris, Average CLIP score: 0.32785840634381186


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01kjwsyj6ye8stmfq6c6b93bdn` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5148, Requested 929. Please try again in 770ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### Loading images from Cloudflare

In [20]:
load_dotenv()

ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID")
ACCESS_KEY = os.getenv("R2_ACCESS_KEY")
SECRET_KEY = os.getenv("R2_SECRET_KEY")
BUCKET_NAME = os.getenv("R2_BUCKET_NAME")

session = boto3.session.Session()
client = session.client(
    "s3",
    region_name="auto",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
)
if is_landmark:
    path = Path.joinpath(Path("data/video_generation/raw/landmark_images/"), NAME.replace(" ", "_").lower())
else:
    path = Path.joinpath(Path("data/video_generation/raw/pharaohs_images/"), NAME.replace(" ", "_").lower())
REMOTE_PREFIX = path

local_frames_dir = Path("temp_frames")
local_frames_dir.mkdir(exist_ok=True)

ordered_local_paths = []

def download_image(idx_key):
    idx, image_key = idx_key
    local_file = local_frames_dir / f"{idx:04d}.jpg"
    client.download_file(BUCKET_NAME, image_key, str(local_file))
    return str(local_file)

with ThreadPoolExecutor(max_workers=8) as executor:
    ordered_local_paths = list(
        executor.map(download_image, enumerate(fetched_images_paths))
    )

In [21]:
image_files = list(local_frames_dir.glob("*.jpg")) + list(local_frames_dir.glob("*.jpeg"))
image_files

[WindowsPath('temp_frames/0000.jpg'),
 WindowsPath('temp_frames/0001.jpg'),
 WindowsPath('temp_frames/0002.jpg'),
 WindowsPath('temp_frames/0003.jpg'),
 WindowsPath('temp_frames/0004.jpg'),
 WindowsPath('temp_frames/0005.jpg'),
 WindowsPath('temp_frames/0006.jpg'),
 WindowsPath('temp_frames/0007.jpg'),
 WindowsPath('temp_frames/0008.jpg'),
 WindowsPath('temp_frames/0009.jpg'),
 WindowsPath('temp_frames/0010.jpg'),
 WindowsPath('temp_frames/0011.jpg')]

In [22]:
from PIL import Image as PILImage
for path in image_files:
    p = Path(path)
    try:
        with PILImage.open(p) as img:
            img = img.convert("RGB")  # handles AVIF, PNG, RGBA, etc.
            img.save(p, "JPEG", quality=95)
        print(f"Converted: {p}")
    except Exception as e:
        print(f"Failed to convert {p}: {e}")

Converted: temp_frames\0000.jpg
Converted: temp_frames\0001.jpg
Converted: temp_frames\0002.jpg
Converted: temp_frames\0003.jpg
Converted: temp_frames\0004.jpg
Converted: temp_frames\0005.jpg
Converted: temp_frames\0006.jpg
Converted: temp_frames\0007.jpg
Converted: temp_frames\0008.jpg
Converted: temp_frames\0009.jpg
Converted: temp_frames\0010.jpg
Converted: temp_frames\0011.jpg


In [23]:
for img_path in Path("temp_frames").glob("*.jpg"):
    print(img_path, img_path.stat().st_size)
    try:
        with Image.open(img_path) as im:
            print("OK:", im.format, im.size)
    except Exception as e:
        print("Broken image:", e)

temp_frames\0000.jpg 113340
OK: JPEG (529, 395)
temp_frames\0001.jpg 315849
OK: JPEG (849, 1390)
temp_frames\0002.jpg 97465
OK: JPEG (522, 345)
temp_frames\0003.jpg 861547
OK: JPEG (1400, 1080)
temp_frames\0004.jpg 163083
OK: JPEG (559, 559)
temp_frames\0005.jpg 273076
OK: JPEG (1200, 675)
temp_frames\0006.jpg 7435932
OK: JPEG (4284, 5712)
temp_frames\0007.jpg 98702
OK: JPEG (594, 391)
temp_frames\0008.jpg 218494
OK: JPEG (875, 577)
temp_frames\0009.jpg 78597
OK: JPEG (546, 362)
temp_frames\0010.jpg 1317111
OK: JPEG (2500, 1667)
temp_frames\0011.jpg 222794
OK: JPEG (798, 590)


### Subtitles

In [24]:
# ---------- SETTINGS ----------
MAX_CHARS_PER_LINE = 35
MAX_LINES = 2
MIN_DURATION = 1.0  # minimum subtitle duration in seconds

# ---------- HELPERS ----------
def normalize_text(text):
    """
    Cleans problematic Unicode characters that break
    subtitle rendering in Pillow / MoviePy.
    """

    # First normalize Unicode composition
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "’": "'",
        "‘": "'",
        "‚": ",",
        "‛": "'",

        "“": '"',
        "”": '"',
        "„": '"',

        "—": "-",
        "–": "-",
        "―": "-",

        "…": "...",

        "\u00A0": " ",   # non-breaking space
        "\u200B": "",    # zero-width space
        "\u200C": "",
        "\u200D": "",
        "\uFEFF": "",    # BOM if embedded
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # Remove any remaining control characters
    text = "".join(
        ch for ch in text
        if unicodedata.category(ch)[0] != "C"
    )

    return text

def format_timestamp(seconds):
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    millis = int((seconds - total_seconds) * 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60

    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"


def split_into_sentences(text):
    return re.split(r'(?<=[.!?]) +', text.strip())


def split_long_text(text, max_chars=MAX_CHARS_PER_LINE):
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        if len(current_line) + len(word) + 1 <= max_chars:
            current_line += (" " if current_line else "") + word
        else:
            lines.append(current_line)
            current_line = word

    if current_line:
        lines.append(current_line)

    # combine into blocks of max 2 lines
    blocks = []
    for i in range(0, len(lines), MAX_LINES):
        block = "\n".join(lines[i:i + MAX_LINES])
        blocks.append(block)

    return blocks

'''
# ---------- MAIN FUNCTION ----------
def generate_srt(paragraphs, durations, output_path="subtitles.srt"):
    assert len(paragraphs) == len(durations), "Paragraphs and durations must match"

    current_time = 0.0
    subtitle_index = 1
    srt_blocks = []

    for paragraph, duration in zip(paragraphs, durations):
        paragraph = normalize_text(paragraph)
        sentences = split_into_sentences(paragraph)

        # break into subtitle chunks
        chunks = []
        for sentence in sentences:
            chunks.extend(split_long_text(sentence))

        total_chars = sum(len(chunk.replace("\n", "")) for chunk in chunks)

        for chunk in chunks:
            chunk_char_count = len(chunk.replace("\n", ""))

            # proportional timing
            chunk_duration = max(
                MIN_DURATION,
                (chunk_char_count / total_chars) * duration
            )

            start_time = current_time
            end_time = current_time + chunk_duration

            srt_block = f"""{subtitle_index}
{format_timestamp(start_time)} --> {format_timestamp(end_time)}
{chunk}

"""
            srt_blocks.append(srt_block)

            current_time = end_time
            subtitle_index += 1

    with open(output_path, "w", encoding="utf-8-sig") as f:
        f.writelines(srt_blocks)

    print(f"SRT file saved to {output_path}")


generate_srt(paragraphs, durations, "Outputs/output_subtitles.srt")'''

'\n# ---------- MAIN FUNCTION ----------\ndef generate_srt(paragraphs, durations, output_path="subtitles.srt"):\n    assert len(paragraphs) == len(durations), "Paragraphs and durations must match"\n\n    current_time = 0.0\n    subtitle_index = 1\n    srt_blocks = []\n\n    for paragraph, duration in zip(paragraphs, durations):\n        paragraph = normalize_text(paragraph)\n        sentences = split_into_sentences(paragraph)\n\n        # break into subtitle chunks\n        chunks = []\n        for sentence in sentences:\n            chunks.extend(split_long_text(sentence))\n\n        total_chars = sum(len(chunk.replace("\n", "")) for chunk in chunks)\n\n        for chunk in chunks:\n            chunk_char_count = len(chunk.replace("\n", ""))\n\n            # proportional timing\n            chunk_duration = max(\n                MIN_DURATION,\n                (chunk_char_count / total_chars) * duration\n            )\n\n            start_time = current_time\n            end_time = cur

In [25]:
def generate_srt(paragraphs, sentences_durations, output_path="subtitles.srt"):
    current_time = 0.0
    subtitle_index = 1
    srt_blocks = []
    duration_index = 0  # pointer into flat sentences_durations list

    for paragraph in paragraphs:
        paragraph = normalize_text(paragraph)
        sentences = split_into_sentences(paragraph)

        for sentence in sentences:
            sentence_duration = sentences_durations[duration_index]
            duration_index += 1

            # Split long sentence into visual chunks
            chunks = split_long_text(sentence)
            total_chars = sum(len(c.replace("\n", "")) for c in chunks)

            for chunk in chunks:
                chunk_char_count = len(chunk.replace("\n", ""))

                # Proportional split only within the sentence now
                chunk_duration = max(
                    MIN_DURATION,
                    (chunk_char_count / total_chars) * sentence_duration
                )

                start_time = current_time
                end_time = current_time + chunk_duration

                srt_block = f"""{subtitle_index}
{format_timestamp(start_time)} --> {format_timestamp(end_time)}
{chunk}

"""
                srt_blocks.append(srt_block)
                current_time = end_time
                subtitle_index += 1

    with open(output_path, "w", encoding="utf-8-sig") as f:
        f.writelines(srt_blocks)

    print(f"SRT file saved to {output_path}")


generate_srt(paragraphs, sentences_durations, "Outputs/output_subtitles.srt")

SRT file saved to Outputs/output_subtitles.srt


### Main Pipeline

In [26]:
def run_ffmpeg(cmd):
    cmd = [str(c) for c in cmd]
    print("Running:", " ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print(result.stderr)  # show FFmpeg error

    if result.returncode != 0:
        raise RuntimeError("FFmpeg failed")

FFMPEG  = shutil.which("ffmpeg")

In [27]:
def _stable_int(seed: str) -> int:
    return int(hashlib.md5(seed.encode("utf-8")).hexdigest()[:8], 16)

def _stable_unit(seed: str) -> float:
    return (_stable_int(seed) % 10_000) / 10_000.0

In [28]:
def plan_kenburns_sequence(
    image_files,
    durations,
    target_w=1920,
    target_h=1080,
    threshold=40,
    min_pan_travel=140,
    max_pan_speed=120,
    vertical_travel_ratio=0.80,
    horizontal_travel_ratio=0.55,
):
    PALETTE = [
        ("zoom", "in"),
        ("pan",  "forward"),
        ("zoom", "out"),
        ("pan",  "reverse"),
    ]

    plans       = []
    palette_idx = 0
    last_sig    = None
    target_ar   = target_w / target_h

    for img_path, duration in zip(image_files, durations):
        img_path = Path(img_path)

        with Image.open(img_path) as img:
            img_w, img_h = img.size

        scale_factor = max(target_w / img_w, target_h / img_h)
        scaled_w     = int(round(img_w * scale_factor))
        scaled_h     = int(round(img_h * scale_factor))

        scaled_w = scaled_w if scaled_w % 2 == 0 else scaled_w + 1
        scaled_h = scaled_h if scaled_h % 2 == 0 else scaled_h + 1
        
        max_x        = max(0, scaled_w - target_w)
        max_y        = max(0, scaled_h - target_h)

        img_ar      = img_w / img_h
        is_vertical = img_ar < target_ar
        pan_axis    = "vertical" if is_vertical else "horizontal"

        travel_x = min(max_x, min(int(max_pan_speed * duration),
                                  int(max_x * horizontal_travel_ratio)))
        travel_y = min(max_y, min(int(max_pan_speed * duration),
                                  int(max_y * vertical_travel_ratio)))
        can_pan  = (travel_y > max(threshold, min_pan_travel)) if is_vertical \
                   else (travel_x > max(threshold, min_pan_travel))

        mode = direction = None
        for attempt in range(len(PALETTE)):
            cand_mode, cand_dir = PALETTE[(palette_idx + attempt) % len(PALETTE)]
            if cand_mode == "pan" and not can_pan:
                continue
            if (cand_mode, cand_dir) == last_sig:
                continue
            mode, direction = cand_mode, cand_dir
            palette_idx = (palette_idx + attempt + 1) % len(PALETTE)
            break

        if mode is None:
            mode        = "zoom"
            direction   = "out" if (last_sig and last_sig[1] == "in") else "in"
            palette_idx = (palette_idx + 1) % len(PALETTE)

        last_sig = (mode, direction)
        plans.append({
            "mode":        mode,
            "direction":   direction,
            "is_vertical": is_vertical,
            "pan_axis":    pan_axis,
        })

    return plans


In [29]:
def create_kenburns_clip(
    image_path,
    duration,
    output_path,
    fps=30,
    target_w=1920,
    target_h=1080,

    threshold=40,
    min_pan_travel=140,
    prefer_pan_when_possible=True,

    zoom_min=1.03,
    zoom_max=1.08,
    zoom_rate_per_sec=0.055,
    zoom_anchor_vertical_y=0.18,
    zoom_anchor_horizontal_y=0.50,

    max_pan_speed=200,
    vertical_travel_ratio=0.80,
    horizontal_travel_ratio=0.55,
    top_bias=0.00,

    motion_scale=1.4,
    use_nvenc=True,

    motion_mode=None,
    motion_direction=None,
):
    total_frames = max(2, int(round(duration * fps)))
    image_path   = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    with Image.open(image_path) as img:
        img_w, img_h = img.size

    scale_factor = max(target_w / img_w, target_h / img_h)
    scaled_w = (int(round(img_w * scale_factor)) )
    scaled_h = (int(round(img_h * scale_factor)) ) 

    scaled_w = scaled_w if scaled_w % 2 == 0 else scaled_w + 1
    scaled_h = scaled_h if scaled_h % 2 == 0 else scaled_h + 1
        
    max_x = max(0, scaled_w - target_w)
    max_y = max(0, scaled_h - target_h)

    denom    = total_frames - 1
    ease_pan  = f"(0.5-0.5*cos(PI*n/{denom}))"
    ease_zoom = f"(0.5-0.5*cos(PI*on/{denom}))"

    target_ar   = target_w / target_h
    img_ar      = img_w / img_h
    is_vertical = img_ar < target_ar

    u = _stable_unit(str(image_path))

    zoom_direction = "out" if u < 0.5 else "in"
    pan_reverse    = u > 0.66

    if motion_direction is not None:
        if motion_direction in ("in", "out"):
            zoom_direction = motion_direction
        elif motion_direction in ("forward", "reverse"):
            pan_reverse = (motion_direction == "reverse")

    dyn_cap = 1.0 + zoom_rate_per_sec * duration
    zmax    = min(zoom_max, dyn_cap)
    z_target = max(zoom_min, min(zoom_min + (zmax - zoom_min) * (0.25 + 0.50 * u), zmax))

    travel_x = min(max_x, min(int(max_pan_speed * duration), int(max_x * horizontal_travel_ratio)))
    travel_y = min(max_y, min(int(max_pan_speed * duration), int(max_y * vertical_travel_ratio)))
    can_pan  = (travel_y > max(threshold, min_pan_travel)) if is_vertical \
               else (travel_x > max(threshold, min_pan_travel))

    ms                     = max(1.0, float(motion_scale))
    mw, mh                 = int(round(target_w * ms)) , int(round(target_h * ms))
    scaled_w2, scaled_h2   = int(round(scaled_w * ms)), int(round(scaled_h * ms))
    max_x2, max_y2         = max(0, scaled_w2 - mw), max(0, scaled_h2 - mh)
    travel_x2, travel_y2   = int(round(min(max_x2, travel_x * ms))),  int(round(min(max_y2, travel_y * ms)))

    def build_pan_vf():
        if is_vertical  and travel_y2 < 220: return None
        if not is_vertical and travel_x2 < 220: return None

        if is_vertical:
            start_y = int(round(max_y2 * top_bias))
            end_y   = min(start_y + travel_y2, max_y2)
            if pan_reverse: start_y, end_y = end_y, start_y
            x_expr = f"{max_x2 // 2}"
            y_expr = f"trunc({start_y}+({end_y}-{start_y})*{ease_pan})"
        else:
            start_x, end_x = 0, travel_x2
            if pan_reverse: start_x, end_x = end_x, start_x
            x_expr = f"trunc({start_x}+({end_x}-{start_x})*{ease_pan})"
            y_expr = f"{max_y2 // 2}"

        return (
            f"scale={scaled_w2}:{scaled_h2}:flags=lanczos,"
            f"crop={mw}:{mh}:x='{x_expr}':y='{y_expr}',"
            f"scale={target_w}:{target_h}:flags=lanczos,"
            f"setsar=1,setdar=16/9,format=yuv420p"
        )

    def build_zoom_vf():
        if zoom_direction == "out":
            z_expr = f"{z_target}-({z_target}-1.0)*{ease_zoom}"
        else:
            z_expr = f"1.0+({z_target}-1.0)*{ease_zoom}"

        ax = 0.5
        ay = zoom_anchor_vertical_y if is_vertical else zoom_anchor_horizontal_y
        x0 = f"max(0,min(({ax})*iw-ow/2,iw-ow))"
        y0 = f"max(0,min(({ay})*ih-oh/2,ih-oh))"

        return (
            f"scale={scaled_w2}:{scaled_h2}:flags=lanczos,"
            f"zoompan=z='{z_expr}':x='if(eq(on,1),{x0},px)':y='if(eq(on,1),{y0},py)'"
            f":d=1:s={mw}x{mh}:fps={fps},"
            f"scale={target_w}:{target_h}:flags=lanczos,"
            f"setsar=1,setdar=16/9,format=yuv420p"
        )

    if motion_mode == "pan":
        vf = build_pan_vf() or build_zoom_vf()
    elif motion_mode == "zoom":
        vf = build_zoom_vf()
    else:
        vf = (build_pan_vf() or build_zoom_vf()) if (prefer_pan_when_possible and can_pan) \
             else build_zoom_vf()

    vcodec = ["-c:v", "h264_nvenc", "-preset", "p1"] if use_nvenc \
             else ["-c:v", "libx264", "-preset", "slow", "-crf", "18"]

    run_ffmpeg([
        "ffmpeg", "-y",
        "-loop", "1", "-framerate", str(fps), "-t", str(duration), "-i", str(image_path),
        "-vf", vf, "-frames:v", str(total_frames),
        *vcodec, "-pix_fmt", "yuv420p","-fps_mode", "cfr",
        output_path,
    ])

In [30]:
def generate_all_clips(image_files, durations, temp_dir="temp_clips"):
    Path(temp_dir).mkdir(exist_ok=True)

    plans = plan_kenburns_sequence(image_files, durations)

    outputs = []
    for i, (img, dur, plan) in enumerate(zip(image_files, durations, plans)):
        out = f"{temp_dir}/clip_{i}.mp4"

        create_kenburns_clip(
            image_path=img,
            duration=dur,
            output_path=out,
            motion_mode=plan["mode"],
            motion_direction=plan["direction"],
        )
        outputs.append(out)

    return outputs

In [31]:
def distribute_durations_exact(seconds_for_chunk, n_clips, fade):
    target_extra = fade * (n_clips - 1)
    extra_for_each = target_extra / n_clips
    seconds_for_chunk = [s + extra_for_each for s in seconds_for_chunk]
    return seconds_for_chunk

def get_duration(path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return float(result.stdout.strip())

In [32]:
def concatenate_clips(clips, output_path, fade=0.45):

    durations_local = [get_duration(c) for c in clips]

    cmd = ["ffmpeg", "-y"]
    for c in clips:
        cmd += ["-i", str(c)]

    filter_parts = []

    # IMPORTANT FIX: normalize every clip before xfade
    filter_parts.append(
        "[0:v]fps=30,scale=1920:1080:flags=lanczos,setsar=1,setdar=16/9,format=yuv420p[v0]"
    )

    for i in range(1, len(clips)):
        filter_parts.append(
            f"[{i}:v]fps=30,scale=1920:1080:flags=lanczos,setsar=1,setdar=16/9,format=yuv420p[v{i}src]"
        )

    cumulative = durations_local[0]
    last = "v0"

    for i in range(1, len(clips)):
        offset = cumulative - fade
        out = f"vx{i}"
        filter_parts.append(
            f"[{last}][v{i}src]xfade=transition=fade:duration={fade}:offset={offset}[{out}]"
        )
        cumulative += durations_local[i] - fade
        last = out

    filter_complex = ";".join(filter_parts)

    cmd += [
        "-safe", "0",
        "-filter_complex", filter_complex,
        "-map", f"[{last}]",
        "-c:v", "h264_nvenc",
        "-preset", "medium",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
      
        str(output_path)
    ]

    run_ffmpeg(cmd)

In [33]:
def add_audio(video_path, audio_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-i", audio_path,
        "-c", "copy",
        "-c:a", "aac",
        output_path
    ]

    run_ffmpeg(cmd)

def add_subtitles(video_path, srt_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-fflags", "+genpts",
        "-i", video_path,
        "-vf", f"subtitles={srt_path}",
        "-c:v", "h264_nvenc",
        "-preset", "p1",
        "-pix_fmt", "yuv420p",
        "-fps_mode", "cfr",
        "-c:a", "copy",
        output_path
    ]

    run_ffmpeg(cmd)

def cleanup_files():
    temp_dir = "temp_clips"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    temp_dir = "temp_frames"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    for f in os.listdir('Outputs'):
        if f.startswith("combined") or f.startswith("with_audio") or f.startswith("concat_list") or f.startswith("output_subtitles") or f.endswith(".wav"):
            os.remove(os.path.join('Outputs', f))

In [34]:
seconds = distribute_durations_exact(
    seconds_for_chunk,
    n_clips=len(image_files),
    fade=0.45
)

# 1. Generate Ken Burns clips
clips = generate_all_clips(image_files, seconds)

# 2. Concatenate
concatenated = "Outputs/combined.mp4"
concatenate_clips(clips, concatenated)

# 3. Add audio
with_audio = "Outputs/with_audio.mp4"
add_audio(concatenated, "Outputs/Final audio.wav", with_audio)

# 4. Add subtitles
final_output = "Outputs/final_video.mp4"
add_subtitles(with_audio, "Outputs/output_subtitles.srt", final_output)

#cleanup_files()

print("Done:", final_output)

Running: ffmpeg -y -loop 1 -framerate 30 -t 4.7325 -i temp_frames\0000.jpg -vf scale=2688:2008:flags=lanczos,zoompan=z='1.0+(1.0475775-1.0)*(0.5-0.5*cos(PI*on/141))':x='if(eq(on,1),max(0,min((0.5)*iw-ow/2,iw-ow)),px)':y='if(eq(on,1),max(0,min((0.18)*ih-oh/2,ih-oh)),py)':d=1:s=2688x1512:fps=30,scale=1920:1080:flags=lanczos,setsar=1,setdar=16/9,format=yuv420p -frames:v 142 -c:v h264_nvenc -preset p1 -pix_fmt yuv420p -fps_mode cfr temp_clips/clip_0.mp4
ffmpeg version 8.0.1-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (Rev8, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom 

In [35]:
cleanup_files()